# Quantum Jam 2026, química cuántica (soluciones)

<img src= "images/quantumjam-logo.svg" />

In [ ]:
!pip install qiskit[visualization] qiskit-aer pyscf

import numpy as np
import matplotlib.pyplot as plt
from math import comb
from scipy.optimize import minimize
from pyscf import gto, scf, fci, mcscf, ao2mo, cc
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import PauliEvolutionGate, efficient_su2
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator, SamplerV2 as AerSampler

# Parte 1

## Ejercicio 1

a. Crea la molécula de $H_2$ con una distancia de 0,735 Å entre los núcleos y la base STO-3G. Muestra la cantidad de electrones y de orbitales.

b. Corre Hartree-Fock y muestra la energía total de la molécula.

c. Muestra la energía de repulsión entre los dos núcleos con el método ```mol.energy_nuc()```. ¿Es positiva o negativa? ¿Por qué?

In [ ]:
mol = gto.M(atom="H 0 0 0; H 0 0 0.735", basis="sto-3g", verbose=0)

print("Cantidad de electrones =", mol.nelectron)
print("Cantidad de orbitales =", mol.nao_nr())

In [ ]:
hf = scf.RHF(mol).run()

print("Energía de Hartree-Fock =", hf.e_tot, "Ha")

In [ ]:
print("Repulsión entre núcleos =", mol.energy_nuc(), "Ha")

**Respuesta.** La energía de repulsión entre los núcleos es positiva, porque los dos protones tienen carga positiva y se repelen. Las energías positivas corresponden a repulsión y las negativas a atracción. La energía total de la molécula es negativa porque la atracción entre los electrones y los núcleos le gana a las repulsiones.

## Ejercicio 2

Repite el cálculo de Hartree-Fock para $H_2$ a 0,735 Å con las bases ```sto-3g```, ```6-31g``` y ```cc-pvdz```. Para cada una, muestra la cantidad de orbitales y la energía de Hartree-Fock.

¿Qué pasa con la energía a medida que la base crece? Si cada orbital cuesta dos qubits, ¿cuántos qubits necesitaríamos con cada base?

In [ ]:
for base in ["sto-3g", "6-31g", "cc-pvdz"]:
    mol = gto.M(atom="H 0 0 0; H 0 0 0.735", basis=base, verbose=0)
    hf = scf.RHF(mol).run()
    norb = mol.nao_nr()
    print(f"{base:8}  orbitales = {norb:2}  qubits = {2 * norb:2}  energía = {hf.e_tot:.6f} Ha")

**Respuesta.** A medida que la base crece, la energía de Hartree-Fock baja, porque con más funciones los orbitales se pueden describir mejor. Pero la cantidad de qubits crece rápido, de 4 qubits con STO-3G a 20 con cc-pVDZ, solo para la molécula más simple que existe.

## Ejercicio 3

a. Calcula la energía de Hartree-Fock y la energía exacta de $H_2$ para 28 distancias entre 0,3 Å y 3 Å, con la base STO-3G. Pueden generar las distancias con ```np.linspace(0.3, 3.0, 28)```.

b. Grafica las dos curvas en un mismo gráfico.

c. Encuentra la distancia del mínimo de cada curva. ¿Coinciden?

d. Grafica el error de Hartree-Fock en mHa en función de la distancia, junto con una línea horizontal en 1,6 mHa. ¿Para qué distancias Hartree-Fock está dentro de la precisión química?

In [ ]:
distancias = np.linspace(0.3, 3.0, 28)
energias_hf, energias_fci = [], []

for d in distancias:
    mol = gto.M(atom=f"H 0 0 0; H 0 0 {d}", basis="sto-3g", verbose=0)
    hf = scf.RHF(mol).run()
    energias_hf.append(hf.e_tot)
    energias_fci.append(fci.FCI(hf).kernel()[0])

energias_hf = np.array(energias_hf)
energias_fci = np.array(energias_fci)

In [ ]:
plt.plot(distancias, energias_hf, "o-", label="Hartree-Fock")
plt.plot(distancias, energias_fci, "o-", label="exacta (FCI)")
plt.xlabel("distancia entre núcleos (Å)")
plt.ylabel("energía (Ha)")
plt.legend()
plt.show()

In [ ]:
print("Mínimo de Hartree-Fock en", distancias[np.argmin(energias_hf)], "Å")
print("Mínimo de FCI en", distancias[np.argmin(energias_fci)], "Å")

In [ ]:
error_hf = 1000 * (energias_hf - energias_fci)

plt.plot(distancias, error_hf, "o-", label="error de Hartree-Fock")
plt.axhline(1.6, color="green", linestyle="--", label="precisión química")
plt.xlabel("distancia entre núcleos (Å)")
plt.ylabel("error (mHa)")
plt.legend()
plt.show()

print("Error mínimo de Hartree-Fock =", error_hf.min(), "mHa")

**Respuesta.** Con esta grilla de distancias, los dos mínimos caen en 0,7 Å. Si usan una grilla más fina, van a ver que el mínimo de Hartree-Fock está un poco antes (cerca de 0,71 Å) que el mínimo exacto (cerca de 0,735 Å). El error de Hartree-Fock nunca baja de la precisión química, ni siquiera en las distancias cortas, y se vuelve enorme cuando estiramos la molécula.

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
def pesos_configuraciones(hf, cantidad=5):
    """Calcula el estado exacto con FCI y devuelve las configuraciones con más peso."""
    norb = hf.mol.nao_nr()
    n_alfa, n_beta = hf.mol.nelec
    energia, ci = fci.FCI(hf).kernel()

    pesos = {}
    for i_alfa in range(ci.shape[0]):
        for i_beta in range(ci.shape[1]):
            # Cada índice representa una forma de acomodar los electrones de un spin
            alfa = fci.cistring.addr2str(norb, n_alfa, i_alfa)
            beta = fci.cistring.addr2str(norb, n_beta, i_beta)
            # Escribimos la configuración con los bits beta a la izquierda y los alfa a la derecha
            bitstring = format(beta, f"0{norb}b") + format(alfa, f"0{norb}b")
            pesos[bitstring] = abs(ci[i_alfa, i_beta]) ** 2

    # Ordenamos de mayor a menor peso
    ordenados = sorted(pesos.items(), key=lambda par: par[1], reverse=True)
    return ordenados[:cantidad]

## Ejercicio 4

a. Usa la función ```pesos_configuraciones``` para ver las configuraciones más importantes de $H_2$ a 0,735 Å y a 3 Å. ¿Qué diferencia notan?

b. Grafica el peso de la configuración de Hartree-Fock (```0101```) en función de la distancia, entre 0,3 Å y 3 Å.

c. Compara este gráfico con el error de Hartree-Fock del ejercicio 3. ¿Qué relación encuentran?

In [ ]:
for d in [0.735, 3.0]:
    mol = gto.M(atom=f"H 0 0 0; H 0 0 {d}", basis="sto-3g", verbose=0)
    hf = scf.RHF(mol).run()
    print(f"Distancia {d} Å")
    for bitstring, peso in pesos_configuraciones(hf):
        print("   ", bitstring, round(peso, 4))

In [ ]:
distancias = np.linspace(0.3, 3.0, 28)
pesos_hf = []

for d in distancias:
    mol = gto.M(atom=f"H 0 0 0; H 0 0 {d}", basis="sto-3g", verbose=0)
    hf = scf.RHF(mol).run()
    pesos_hf.append(dict(pesos_configuraciones(hf, cantidad=4))["0101"])

plt.plot(distancias, pesos_hf, "o-")
plt.xlabel("distancia entre núcleos (Å)")
plt.ylabel("peso de la configuración 0101")
plt.show()

**Respuesta.** Cerca de la distancia de enlace, casi todo el peso está en la configuración de Hartree-Fock ```0101```, y el resto de las configuraciones tiene un peso muy chico. A 3 Å, en cambio, el peso se reparte casi en partes iguales entre ```0101``` y ```1010```, la configuración con los dos electrones en el orbital antienlazante.

Cuanto menos peso tiene la configuración de Hartree-Fock, más grande es su error, porque Hartree-Fock describe el estado con esa única configuración.

## Ejercicio 5

a. Escribe una función ```cantidad_configuraciones(n, n_alfa, n_beta)``` que devuelva la cantidad de configuraciones válidas.

b. Para cada una de las siguientes moléculas, con la base STO-3G, calcula la cantidad de orbitales, la cantidad de configuraciones válidas y la cantidad total de bitstrings. Pueden obtener la cantidad de electrones de cada spin con ```mol.nelec```.

- $H_2$, con ```"H 0 0 0; H 0 0 0.735"```
- Una cadena de cuatro átomos de hidrógeno en línea, $H_4$, con ```"H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3"```
- Agua, $H_2O$, con ```"O 0 0 0.1173; H 0 0.7572 -0.4692; H 0 -0.7572 -0.4692"```
- Nitrógeno, $N_2$, con ```"N 0 0 0; N 0 0 1.1"```

c. ¿Qué porcentaje de los bitstrings son configuraciones válidas en cada caso?

In [ ]:
def cantidad_configuraciones(n, n_alfa, n_beta):
    return comb(n, n_alfa) * comb(n, n_beta)

In [ ]:
moleculas = {
    "H2": "H 0 0 0; H 0 0 0.735",
    "H4": "H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3",
    "H2O": "O 0 0 0.1173; H 0 0.7572 -0.4692; H 0 -0.7572 -0.4692",
    "N2": "N 0 0 0; N 0 0 1.1",
}

for nombre, atomos in moleculas.items():
    mol = gto.M(atom=atomos, basis="sto-3g", verbose=0)
    n = mol.nao_nr()
    n_alfa, n_beta = mol.nelec
    validas = cantidad_configuraciones(n, n_alfa, n_beta)
    print(f"{nombre:4} orbitales = {n:2}  configuraciones = {validas:6}  bitstrings = {2 ** (2 * n)}")

In [ ]:
for nombre, atomos in moleculas.items():
    mol = gto.M(atom=atomos, basis="sto-3g", verbose=0)
    n = mol.nao_nr()
    n_alfa, n_beta = mol.nelec
    porcentaje = 100 * cantidad_configuraciones(n, n_alfa, n_beta) / 2 ** (2 * n)
    print(f"{nombre:4} {porcentaje:.4f} %")

**Respuesta.** La fracción de bitstrings válidos se vuelve cada vez más chica a medida que la molécula crece. Pasa de un 25 % en $H_2$ a apenas un 1,4 % en $N_2$, que tiene solo 10 orbitales. Vamos a volver sobre esta idea en la parte 2.

## Ejercicio 6

a. Crea el operador de destrucción de un qubit, $a = \frac{X+iY}{2}$, con ```SparsePauliOp```.

b. Aplica el operador a los estados $\left|1\right>$ y $\left|0\right>$. Para eso, pueden multiplicar la matriz del operador (```.to_matrix()```) por el vector de amplitudes del estado (```Statevector.from_label("1").data```) con el operador ```@``` de Python. ¿Qué obtienen?

c. Obtén el operador de creación $a^\dagger$ con el método ```.adjoint()``` y calcula el producto $a^\dagger a$ con el método ```.dot()```. Muestra que es igual a $\frac{I - Z}{2}$. Este operador se llama **operador número**, porque vale 1 si hay un electrón y 0 si no lo hay.

In [ ]:
a = SparsePauliOp(["X", "Y"], [0.5, 0.5j])
print(a)

In [ ]:
print("a|1> =", a.to_matrix() @ Statevector.from_label("1").data)
print("a|0> =", a.to_matrix() @ Statevector.from_label("0").data)

In [ ]:
numero = a.adjoint().dot(a).simplify()
print(numero)
print(numero.to_matrix())

print(SparsePauliOp(["I", "Z"], [0.5, -0.5]).to_matrix())

**Respuesta.** El operador lleva $\left|1\right>$ a $\left|0\right>$ y anula a $\left|0\right>$ (el resultado es el vector nulo). El producto $a^\dagger a$ da $0{,}5\,I - 0{,}5\,Z$, que tiene la matriz $\left|1\right>\left<1\right|$. Por eso vale 1 cuando el qubit está ocupado y 0 cuando está vacío.

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
def molecula(atomos, base="sto-3g", n_congelados=0):
    """Calcula Hartree-Fock, la energía exacta y las integrales de una molécula."""
    mol = gto.M(atom=atomos, basis=base, verbose=0)
    hf = scf.RHF(mol).run()

    # Los orbitales congelados no entran en el problema, ni tampoco sus electrones
    norb = mol.nao_nr() - n_congelados
    n_electrones = mol.nelectron - 2 * n_congelados

    # PySCF calcula las integrales h y g solo para los orbitales activos
    activo = mcscf.CASCI(hf, norb, n_electrones)
    h, constante = activo.get_h1cas()
    g = ao2mo.restore(1, activo.get_h2cas(), norb)

    return {
        "hf": hf,
        "norb": norb,
        "nelec": (n_electrones // 2, n_electrones // 2),
        "h": h,
        "g": g,
        "constante": constante,
        "e_hf": hf.e_tot,
        "e_fci": activo.kernel()[0],
    }


def destruccion_jordan_wigner(p, n_qubits):
    """Operador de destrucción del spin-orbital p con el mapeo de Jordan-Wigner."""
    # Z en los qubits 0, 1, ..., p-1 y (X + iY)/2 en el qubit p
    return SparsePauliOp.from_sparse_list(
        [("Z" * p + "X", list(range(p + 1)), 0.5),
         ("Z" * p + "Y", list(range(p + 1)), 0.5j)],
        n_qubits,
    )


def hamiltoniano_qubits(m, destruccion=destruccion_jordan_wigner):
    """Arma el Hamiltoniano de la molécula m como una suma de operadores de Pauli."""
    norb = m["norb"]
    n_qubits = 2 * norb
    h, g = m["h"], m["g"]

    # Operadores de destrucción y creación de cada spin-orbital
    # Los qubits 0, ..., norb-1 son los spin-orbitales alfa y los siguientes son los beta
    a = [destruccion(p, n_qubits) for p in range(n_qubits)]
    a_dag = [op.adjoint() for op in a]

    # salto[s][p][q] mueve un electrón con spin s del orbital q al orbital p
    salto = [[[a_dag[s * norb + p].dot(a[s * norb + q]).simplify()
               for q in range(norb)] for p in range(norb)] for s in range(2)]

    # Empezamos con la parte constante (repulsión entre núcleos y orbitales congelados)
    terminos = [SparsePauliOp("I" * n_qubits, m["constante"])]

    # Saltos de un electrón
    for s in range(2):
        for p in range(norb):
            for q in range(norb):
                terminos.append(h[p, q] * salto[s][p][q])

    # Saltos de dos electrones a la vez
    for s in range(2):
        for t in range(2):
            for p in range(norb):
                for q in range(norb):
                    segundo_salto = SparsePauliOp.sum(
                        [g[p, q, r, u] * salto[t][r][u] for r in range(norb) for u in range(norb)])
                    terminos.append(0.5 * salto[s][p][q].dot(segundo_salto))
                    if s == t:
                        # Corrección por el orden de los operadores cuando los dos electrones tienen el mismo spin
                        for u in range(norb):
                            terminos.append(-0.5 * g[p, q, q, u] * salto[s][p][u])

    return SparsePauliOp.sum(terminos).simplify(atol=1e-10)

## Ejercicio 7

a. Arma el Hamiltoniano de qubits de $H_2$ a 0,735 Å. ¿Cuántos qubits tiene? ¿Cuántos términos de Pauli?

b. Convierte el Hamiltoniano en una matriz con ```.to_matrix()``` y calcula su autovalor más chico con ```np.linalg.eigvalsh()```. Compara el resultado con la energía exacta ```m["e_fci"]```.

c. Calcula el valor esperado del Hamiltoniano en el estado de Hartree-Fock, ```Statevector.from_label("0101")```, con el método ```.expectation_value()```. Compara el resultado con la energía de Hartree-Fock ```m["e_hf"]```.

In [ ]:
m = molecula("H 0 0 0; H 0 0 0.735")
H = hamiltoniano_qubits(m)

print("Cantidad de qubits =", H.num_qubits)
print("Cantidad de términos =", len(H))

In [ ]:
autovalores = np.linalg.eigvalsh(H.to_matrix())

print("Autovalor más chico =", autovalores[0])
print("Energía exacta      =", m["e_fci"])

In [ ]:
estado_hf = Statevector.from_label("0101")

print("Valor esperado en 0101 =", estado_hf.expectation_value(H).real)
print("Energía de Hartree-Fock =", m["e_hf"])

**Respuesta.** El Hamiltoniano tiene 4 qubits y 15 términos. Su autovalor más chico coincide con la energía exacta, y el valor esperado en el estado ```0101``` coincide con la energía de Hartree-Fock.

## Ejercicio 8

a. Crea la molécula de agua (con la geometría del ejercicio 5) sin congelar orbitales y congelando un orbital. Para cada caso, muestra la cantidad de qubits que necesitaríamos y la energía exacta.

b. ¿Cuánto cambia la energía exacta al congelar el orbital? ¿Es un cambio mayor o menor que la precisión química?

c. Prueba congelando dos orbitales. ¿Qué pasa con el error? ¿Por qué creen que pasa eso?

In [ ]:
agua = "O 0 0 0.1173; H 0 0.7572 -0.4692; H 0 -0.7572 -0.4692"

completa = molecula(agua)
congelada = molecula(agua, n_congelados=1)

print("Sin congelar   qubits =", 2 * completa["norb"], " energía =", completa["e_fci"])
print("Un congelado   qubits =", 2 * congelada["norb"], " energía =", congelada["e_fci"])

In [ ]:
print("Diferencia =", 1000 * (congelada["e_fci"] - completa["e_fci"]), "mHa")

In [ ]:
dos = molecula(agua, n_congelados=2)

print("Dos congelados  qubits =", 2 * dos["norb"])
print("Diferencia =", 1000 * (dos["e_fci"] - completa["e_fci"]), "mHa")

**Respuesta.** Congelar el orbital 1s del oxígeno ahorra dos qubits y cambia la energía muy poco, bastante menos que la precisión química. Al congelar un segundo orbital, el error se vuelve mucho más grande, porque ese orbital ya no es un orbital interno. Sus electrones sí participan de los enlaces, así que no se los puede dejar fijos.

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
def destruccion_paridad(p, n_qubits):
    """Operador de destrucción del spin-orbital p con el mapeo de paridad."""
    resto = list(range(p + 1, n_qubits))
    if p == 0:
        local = [("X", [0], 0.5), ("Y", [0], 0.5j)]
    else:
        local = [("ZX", [p - 1, p], 0.5), ("Y", [p], 0.5j)]
    # Además, X en todos los qubits que vienen después de p
    return SparsePauliOp.from_sparse_list(
        [(letras + "X" * len(resto), qubits + resto, coef) for letras, qubits, coef in local],
        n_qubits,
    )


def reducir_dos_qubits(H, m):
    """Elimina los dos qubits cuyo valor conocemos por las paridades de los electrones."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    # Qubit que guarda la paridad de los electrones alfa y qubit que guarda la paridad total
    qubits_conocidos = {norb - 1: (-1) ** n_alfa, 2 * norb - 1: (-1) ** (n_alfa + n_beta)}
    quedan = [k for k in range(2 * norb) if k not in qubits_conocidos]

    etiquetas, coeficientes = [], []
    for etiqueta, coef in H.to_list():
        letras = etiqueta[::-1]  # Damos vuelta la etiqueta para que letras[k] sea el qubit k
        for k, valor in qubits_conocidos.items():
            if letras[k] == "Z":
                coef = coef * valor  # Reemplazamos Z por su valor, +1 o -1
        etiquetas.append("".join(letras[k] for k in quedan)[::-1])
        coeficientes.append(coef)

    return SparsePauliOp(etiquetas, coeficientes).simplify()

## Ejercicio 9

a. Arma el Hamiltoniano de la cadena $H_4$ (con la geometría del ejercicio 5) con Jordan-Wigner y con el mapeo de paridad reducido. ¿Cuántos qubits tiene cada uno?

b. Calcula el autovalor más chico de los dos Hamiltonianos y compáralos con la energía exacta.

c. Combina los dos trucos en la molécula de agua, congelando un orbital y usando la reducción de dos qubits. ¿Cuántos qubits quedan, comparado con los 14 que necesitábamos al principio?

In [ ]:
m = molecula("H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3")
H_jw = hamiltoniano_qubits(m)
H_red = reducir_dos_qubits(hamiltoniano_qubits(m, destruccion=destruccion_paridad), m)

print("Jordan-Wigner     qubits =", H_jw.num_qubits)
print("Paridad reducida  qubits =", H_red.num_qubits)

In [ ]:
print("Jordan-Wigner     ", np.linalg.eigvalsh(H_jw.to_matrix())[0])
print("Paridad reducida  ", np.linalg.eigvalsh(H_red.to_matrix())[0])
print("Energía exacta    ", m["e_fci"])

In [ ]:
agua = molecula("O 0 0 0.1173; H 0 0.7572 -0.4692; H 0 -0.7572 -0.4692", n_congelados=1)
H_agua = reducir_dos_qubits(hamiltoniano_qubits(agua, destruccion=destruccion_paridad), agua)

print("Qubits para el agua =", H_agua.num_qubits)

**Respuesta.** Para $H_4$ pasamos de 8 a 6 qubits, con exactamente la misma energía. Para el agua, congelando un orbital y usando la reducción de dos qubits pasamos de 14 a 10 qubits.

## Ejercicio 10

Para $H_2$, la cadena $H_4$ y el agua con un orbital congelado, muestra la cantidad de qubits, la cantidad de términos del Hamiltoniano y la cantidad de grupos que conmutan qubit a qubit. ¿Cuántos circuitos nos ahorramos en cada caso? ¿Cómo crece la cantidad de términos con la cantidad de qubits?

In [ ]:
casos = {
    "H2": molecula("H 0 0 0; H 0 0 0.735"),
    "H4": molecula("H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3"),
    "H2O": molecula("O 0 0 0.1173; H 0 0.7572 -0.4692; H 0 -0.7572 -0.4692", n_congelados=1),
}

for nombre, m in casos.items():
    H = hamiltoniano_qubits(m)
    grupos = H.group_commuting(qubit_wise=True)
    print(f"{nombre:4} qubits = {H.num_qubits:2}  términos = {len(H):4}  grupos = {len(grupos):3}")

**Respuesta.** Agrupar reduce mucho la cantidad de circuitos. En estos ejemplos necesitamos entre 3 y 5 veces menos circuitos que términos, y el ahorro es mayor cuanto más grande es la molécula. La cantidad de términos crece muy rápido con la cantidad de qubits, aproximadamente como la cuarta potencia, porque el término de dos electrones tiene cuatro índices.

# Parte 2

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
def molecula(atomos, base="sto-3g", n_congelados=0):
    """Calcula Hartree-Fock, la energía exacta y las integrales de una molécula."""
    mol = gto.M(atom=atomos, basis=base, verbose=0)
    hf = scf.RHF(mol).run()

    # Los orbitales congelados no entran en el problema, ni tampoco sus electrones
    norb = mol.nao_nr() - n_congelados
    n_electrones = mol.nelectron - 2 * n_congelados

    # PySCF calcula las integrales h y g solo para los orbitales activos
    activo = mcscf.CASCI(hf, norb, n_electrones)
    h, constante = activo.get_h1cas()
    g = ao2mo.restore(1, activo.get_h2cas(), norb)

    return {
        "hf": hf,
        "norb": norb,
        "nelec": (n_electrones // 2, n_electrones // 2),
        "h": h,
        "g": g,
        "constante": constante,
        "e_hf": hf.e_tot,
        "e_fci": activo.kernel()[0],
    }


def destruccion_jordan_wigner(p, n_qubits):
    """Operador de destrucción del spin-orbital p con el mapeo de Jordan-Wigner."""
    # Z en los qubits 0, 1, ..., p-1 y (X + iY)/2 en el qubit p
    return SparsePauliOp.from_sparse_list(
        [("Z" * p + "X", list(range(p + 1)), 0.5),
         ("Z" * p + "Y", list(range(p + 1)), 0.5j)],
        n_qubits,
    )


def hamiltoniano_qubits(m, destruccion=destruccion_jordan_wigner):
    """Arma el Hamiltoniano de la molécula m como una suma de operadores de Pauli."""
    norb = m["norb"]
    n_qubits = 2 * norb
    h, g = m["h"], m["g"]

    # Operadores de destrucción y creación de cada spin-orbital
    # Los qubits 0, ..., norb-1 son los spin-orbitales alfa y los siguientes son los beta
    a = [destruccion(p, n_qubits) for p in range(n_qubits)]
    a_dag = [op.adjoint() for op in a]

    # salto[s][p][q] mueve un electrón con spin s del orbital q al orbital p
    salto = [[[a_dag[s * norb + p].dot(a[s * norb + q]).simplify()
               for q in range(norb)] for p in range(norb)] for s in range(2)]

    # Empezamos con la parte constante (repulsión entre núcleos y orbitales congelados)
    terminos = [SparsePauliOp("I" * n_qubits, m["constante"])]

    # Saltos de un electrón
    for s in range(2):
        for p in range(norb):
            for q in range(norb):
                terminos.append(h[p, q] * salto[s][p][q])

    # Saltos de dos electrones a la vez
    for s in range(2):
        for t in range(2):
            for p in range(norb):
                for q in range(norb):
                    segundo_salto = SparsePauliOp.sum(
                        [g[p, q, r, u] * salto[t][r][u] for r in range(norb) for u in range(norb)])
                    terminos.append(0.5 * salto[s][p][q].dot(segundo_salto))
                    if s == t:
                        # Corrección por el orden de los operadores cuando los dos electrones tienen el mismo spin
                        for u in range(norb):
                            terminos.append(-0.5 * g[p, q, q, u] * salto[s][p][u])

    return SparsePauliOp.sum(terminos).simplify(atol=1e-10)


def destruccion_paridad(p, n_qubits):
    """Operador de destrucción del spin-orbital p con el mapeo de paridad."""
    resto = list(range(p + 1, n_qubits))
    if p == 0:
        local = [("X", [0], 0.5), ("Y", [0], 0.5j)]
    else:
        local = [("ZX", [p - 1, p], 0.5), ("Y", [p], 0.5j)]
    # Además, X en todos los qubits que vienen después de p
    return SparsePauliOp.from_sparse_list(
        [(letras + "X" * len(resto), qubits + resto, coef) for letras, qubits, coef in local],
        n_qubits,
    )


def reducir_dos_qubits(H, m):
    """Elimina los dos qubits cuyo valor conocemos por las paridades de los electrones."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    # Qubit que guarda la paridad de los electrones alfa y qubit que guarda la paridad total
    qubits_conocidos = {norb - 1: (-1) ** n_alfa, 2 * norb - 1: (-1) ** (n_alfa + n_beta)}
    quedan = [k for k in range(2 * norb) if k not in qubits_conocidos]

    etiquetas, coeficientes = [], []
    for etiqueta, coef in H.to_list():
        letras = etiqueta[::-1]  # Damos vuelta la etiqueta para que letras[k] sea el qubit k
        for k, valor in qubits_conocidos.items():
            if letras[k] == "Z":
                coef = coef * valor  # Reemplazamos Z por su valor, +1 o -1
        etiquetas.append("".join(letras[k] for k in quedan)[::-1])
        coeficientes.append(coef)

    return SparsePauliOp(etiquetas, coeficientes).simplify()

## Ejercicio 1

Consideren el Hamiltoniano de un qubit $\hat{H} = Z + 0{,}5\,X$.

a. Crea el Hamiltoniano con ```SparsePauliOp``` y calcula su autovalor más chico con ```np.linalg.eigvalsh()```.

b. Para 100 valores de $\theta$ entre $0$ y $2\pi$, prepara el estado $R_Y(\theta)\left|0\right>$ con un circuito y calcula el valor esperado de $\hat{H}$ (pueden usar ```Statevector``` y el método ```.expectation_value()```).

c. Grafica la energía en función de $\theta$, junto con una línea horizontal en el autovalor más chico. ¿La curva queda alguna vez por debajo de la línea? ¿Para qué valor de $\theta$ se alcanza el mínimo?

In [ ]:
H = SparsePauliOp(["Z", "X"], [1, 0.5])
e_0 = np.linalg.eigvalsh(H.to_matrix())[0]

print("Autovalor más chico =", e_0)

In [ ]:
thetas = np.linspace(0, 2 * np.pi, 100)
energias = []

for theta in thetas:
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    energias.append(Statevector(qc).expectation_value(H).real)

In [ ]:
plt.plot(thetas, energias, label="valor esperado")
plt.axhline(e_0, color="red", linestyle="--", label="autovalor más chico")
plt.xlabel("θ")
plt.ylabel("energía")
plt.legend()
plt.show()

print("Mínimo encontrado =", min(energias), "en θ =", thetas[np.argmin(energias)])

**Respuesta.** La curva nunca queda por debajo del autovalor más chico, como dice el principio variacional. La toca en un solo punto, cerca de $\theta = 3{,}6$, donde el estado $R_Y(\theta)\left|0\right>$ coincide con el estado fundamental.

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
def vqe(ansatz, H, metodo="COBYLA", x0=None, estimator=None, maxiter=3000):
    """Busca los parámetros del ansatz que minimizan la energía de H."""
    if estimator is None:
        estimator = StatevectorEstimator()
    if x0 is None:
        x0 = np.zeros(ansatz.num_parameters)

    historial = []

    def energia(parametros):
        # La computadora cuántica (acá simulada) mide la energía para estos parámetros
        valor = float(estimator.run([(ansatz, H, parametros)]).result()[0].data.evs)
        historial.append(valor)
        return valor

    # El optimizador clásico propone parámetros nuevos hasta que la energía deja de bajar
    resultado = minimize(energia, x0, method=metodo, options={"maxiter": maxiter})
    return resultado, historial


def contar_cnots(circuito):
    """Cuenta las compuertas CNOT que quedan al transpilar el circuito."""
    transpilado = transpile(circuito, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)
    return transpilado.count_ops().get("cx", 0)

## Ejercicio 2

a. Arma el Hamiltoniano de $H_2$ a 0,735 Å con el mapeo de paridad reducido (que tiene 2 qubits).

b. Crea el ansatz ```efficient_su2(2, reps=1)```. ¿Cuántos parámetros tiene? Pueden verlo con ```ansatz.num_parameters```.

c. Corre la función ```vqe``` con parámetros iniciales chicos y aleatorios. Pueden generarlos con ```np.random.default_rng(0).uniform(-0.1, 0.1, ansatz.num_parameters)```.

d. Grafica la energía en cada paso del optimizador (la lista ```historial```) junto con una línea horizontal en la energía exacta. Muestra el error final en mHa.

In [ ]:
m = molecula("H 0 0 0; H 0 0 0.735")
H_red = reducir_dos_qubits(hamiltoniano_qubits(m, destruccion=destruccion_paridad), m)

In [ ]:
ansatz = efficient_su2(2, reps=1)
print("Parámetros =", ansatz.num_parameters)

In [ ]:
x0 = np.random.default_rng(0).uniform(-0.1, 0.1, ansatz.num_parameters)
resultado, historial = vqe(ansatz, H_red, x0=x0)

In [ ]:
plt.plot(historial, label="VQE")
plt.axhline(m["e_fci"], color="red", linestyle="--", label="energía exacta")
plt.xlabel("paso del optimizador")
plt.ylabel("energía (Ha)")
plt.legend()
plt.show()

print("Error final =", 1000 * (resultado.fun - m["e_fci"]), "mHa")

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
def saltos_posibles(m):
    """Lista todos los saltos de uno y de dos electrones desde Hartree-Fock."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    # Spin-orbitales ocupados y vacíos en Hartree-Fock, para cada spin
    ocupados = [list(range(n_alfa)), [norb + i for i in range(n_beta)]]
    vacios = [list(range(n_alfa, norb)), [norb + i for i in range(n_beta, norb)]]

    saltos = []
    # Un electrón salta de i a a, sin cambiar su spin
    for s in range(2):
        for i in ocupados[s]:
            for a in vacios[s]:
                saltos.append(((i,), (a,)))
    # Dos electrones de distinto spin saltan a la vez
    for i in ocupados[0]:
        for j in ocupados[1]:
            for a in vacios[0]:
                for b in vacios[1]:
                    saltos.append(((i, j), (a, b)))
    # Dos electrones del mismo spin saltan a la vez
    for s in range(2):
        for x, i in enumerate(ocupados[s]):
            for j in ocupados[s][x + 1:]:
                for y, a in enumerate(vacios[s]):
                    for b in vacios[s][y + 1:]:
                        saltos.append(((i, j), (a, b)))
    return saltos


def ansatz_ucc(m):
    """Ansatz químico que parte de Hartree-Fock y agrega un ángulo por cada salto posible."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    n_qubits = 2 * norb
    a = [destruccion_jordan_wigner(p, n_qubits) for p in range(n_qubits)]
    saltos = saltos_posibles(m)
    theta = ParameterVector("θ", len(saltos))

    circuito = QuantumCircuit(n_qubits)
    # Estado de Hartree-Fock, con X en los spin-orbitales ocupados
    for i in range(n_alfa):
        circuito.x(i)
    for i in range(n_beta):
        circuito.x(norb + i)

    for angulo, (desde, hacia) in zip(theta, saltos):
        # T saca los electrones de "desde" y los pone en "hacia"
        T = SparsePauliOp("I" * n_qubits)
        for p in hacia:
            T = T.dot(a[p].adjoint())
        for p in reversed(desde):
            T = T.dot(a[p])
        # G = i(T - T†) es hermítico, y la compuerta aplica exp(θ(T - T†))
        G = (1j * (T - T.adjoint())).simplify()
        circuito.append(PauliEvolutionGate(G, time=angulo), range(n_qubits))

    # Transpilamos una sola vez para que la simulación sea rápida
    return transpile(circuito, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)

## Ejercicio 3

a. Crea el ansatz químico de $H_2$ a 0,735 Å (con 4 qubits y Jordan-Wigner) y corre VQE empezando con todos los parámetros en cero, es decir, desde Hartree-Fock.

b. Compara los dos ansatz (el del ejercicio 2 y este) mostrando la cantidad de qubits, de parámetros y de CNOTs, la cantidad de veces que el optimizador tuvo que medir la energía (```resultado.nfev```) y el error final en mHa.

c. ¿Qué ventajas y desventajas tiene cada ansatz?

In [ ]:
m = molecula("H 0 0 0; H 0 0 0.735")
H = hamiltoniano_qubits(m)
ansatz_quimico = ansatz_ucc(m)
resultado_quimico, historial_quimico = vqe(ansatz_quimico, H)

In [ ]:
H_red = reducir_dos_qubits(hamiltoniano_qubits(m, destruccion=destruccion_paridad), m)
ansatz_generico = efficient_su2(2, reps=1)
x0 = np.random.default_rng(0).uniform(-0.1, 0.1, ansatz_generico.num_parameters)
resultado_generico, historial_generico = vqe(ansatz_generico, H_red, x0=x0)

for nombre, ansatz, resultado in [("genérico", ansatz_generico, resultado_generico),
                                  ("químico", ansatz_quimico, resultado_quimico)]:
    error = 1000 * (resultado.fun - m["e_fci"])
    print(f"{nombre:9} qubits = {ansatz.num_qubits}  parámetros = {ansatz.num_parameters:2}  "
          f"CNOTs = {contar_cnots(ansatz):2}  mediciones = {resultado.nfev:4}  error = {error:.6f} mHa")

**Respuesta.** Los dos ansatz llegan a la energía exacta. El ansatz genérico usa la mitad de qubits (gracias a la reducción de dos qubits) y una sola CNOT, pero tiene más parámetros y el optimizador necesita muchas más mediciones de la energía. El ansatz químico tiene pocos parámetros y arranca desde Hartree-Fock, así que converge enseguida, pero su circuito tiene bastantes más CNOTs, y en una computadora cuántica real eso significa más ruido. En moléculas más grandes, además, el ansatz genérico suele tener problemas para encontrar el mínimo, como vamos a ver en el ejercicio 6.

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
# Estimator con ruido de medición, con una semilla para que los resultados se puedan repetir
estimator_ruidoso = AerEstimator(options={"default_precision": 0.01, "run_options": {"seed": 5}})

## Ejercicio 4

a. Usando el ansatz químico de $H_2$ a 0,735 Å, corre VQE con los optimizadores ```COBYLA```, ```Powell``` y ```L-BFGS-B``` usando el ```StatevectorEstimator```. Para cada uno, muestra el error final y la cantidad de mediciones.

b. Repite el inciso anterior usando el ```estimator_ruidoso```, con un máximo de 300 iteraciones (```maxiter=300```). Para que la comparación sea justa, calcula el error final evaluando los parámetros que encontró el optimizador (```resultado.x```) con el ```StatevectorEstimator```, que es exacto.

c. ¿Qué optimizador elegirían para una computadora cuántica real? ¿Por qué?

In [ ]:
m = molecula("H 0 0 0; H 0 0 0.735")
H = hamiltoniano_qubits(m)
ansatz = ansatz_ucc(m)

for metodo in ["COBYLA", "Powell", "L-BFGS-B"]:
    resultado, historial = vqe(ansatz, H, metodo=metodo)
    print(f"{metodo:9} error = {1000 * (resultado.fun - m['e_fci']):.6f} mHa  mediciones = {resultado.nfev}")

In [ ]:
exacto = StatevectorEstimator()

for metodo in ["COBYLA", "Powell", "L-BFGS-B"]:
    resultado, historial = vqe(ansatz, H, metodo=metodo, estimator=estimator_ruidoso, maxiter=300)
    energia_real = exacto.run([(ansatz, H, resultado.x)]).result()[0].data.evs
    print(f"{metodo:9} error = {1000 * (energia_real - m['e_fci']):.3f} mHa  mediciones = {resultado.nfev}")

**Respuesta.** Sin ruido, los tres optimizadores llegan a la energía exacta, y ```L-BFGS-B``` es el que necesita menos mediciones. Con ruido, en cambio, ```L-BFGS-B``` ni siquiera se mueve de Hartree-Fock (su error es exactamente el error de Hartree-Fock), porque el ruido arruina su estimación del gradiente. Los optimizadores que no usan el gradiente se defienden mejor, aunque con tanto ruido tampoco llegan siempre a la precisión química, y el resultado cambia bastante si cambian la semilla. En una computadora cuántica real se suelen usar optimizadores que no necesitan el gradiente o que están pensados especialmente para funciones con ruido, como SPSA, y se aumenta la cantidad de shots a medida que nos acercamos al mínimo.

## Ejercicio 5

a. Calcula la energía de $H_2$ con VQE (usando el ansatz químico) para 15 distancias entre 0,3 Å y 3 Å. Para cada distancia, guarda también la energía de Hartree-Fock y la energía exacta. Usen los parámetros óptimos de cada distancia como punto de partida para la siguiente.

b. Grafica las tres curvas.

c. Grafica el error de VQE y el de Hartree-Fock en mHa, con escala logarítmica en el eje vertical (```plt.yscale("log")```), junto con una línea horizontal en la precisión química. ¿VQE logra precisión química en toda la curva?

In [ ]:
distancias = np.linspace(0.3, 3.0, 15)
energias_vqe, energias_hf, energias_fci = [], [], []
x0 = None

for d in distancias:
    m = molecula(f"H 0 0 0; H 0 0 {d}")
    H = hamiltoniano_qubits(m)
    resultado, historial = vqe(ansatz_ucc(m), H, x0=x0)
    x0 = resultado.x  # punto de partida para la siguiente distancia
    energias_vqe.append(resultado.fun)
    energias_hf.append(m["e_hf"])
    energias_fci.append(m["e_fci"])

energias_vqe, energias_hf, energias_fci = map(np.array, (energias_vqe, energias_hf, energias_fci))

In [ ]:
plt.plot(distancias, energias_hf, "o-", label="Hartree-Fock")
plt.plot(distancias, energias_fci, "-", label="exacta (FCI)")
plt.plot(distancias, energias_vqe, "x", markersize=10, label="VQE")
plt.xlabel("distancia entre núcleos (Å)")
plt.ylabel("energía (Ha)")
plt.legend()
plt.show()

In [ ]:
# Usamos un valor mínimo para que la escala logarítmica funcione aunque el error sea cero
error_vqe = np.maximum(1000 * (energias_vqe - energias_fci), 1e-8)
error_hf = 1000 * (energias_hf - energias_fci)

plt.plot(distancias, error_hf, "o-", label="Hartree-Fock")
plt.plot(distancias, error_vqe, "x-", label="VQE")
plt.axhline(1.6, color="green", linestyle="--", label="precisión química")
plt.yscale("log")
plt.xlabel("distancia entre núcleos (Å)")
plt.ylabel("error (mHa)")
plt.legend()
plt.show()

**Respuesta.** VQE con el ansatz químico logra precisión química en toda la curva, incluso donde Hartree-Fock tiene errores enormes. Para $H_2$ en la base STO-3G el ansatz químico puede representar el estado exacto, así que el error es prácticamente cero. En moléculas más grandes esto ya no es así.

## Ejercicio 6

Tomen la cadena $H_4$ con los átomos separados por 1 Å, ```"H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3"```. Prueben, al menos, estas combinaciones.

- Jordan-Wigner (8 qubits) con el ansatz químico, empezando desde Hartree-Fock.
- Paridad reducida (6 qubits) con ```efficient_su2``` y ```reps``` igual a 1, 2 y 3, empezando con parámetros chicos y aleatorios y con ```maxiter=5000```.

Para cada combinación, muestren la cantidad de qubits, de parámetros, de CNOTs y de grupos de medición del Hamiltoniano, la cantidad de mediciones de la energía que necesitó el optimizador y el error final en mHa.

¿Alguna combinación logra precisión química? Si tuvieran que correr este cálculo en una computadora cuántica real, ¿cuál elegirían y por qué? ¿Se les ocurre alguna otra combinación que valga la pena probar?

In [ ]:
m = molecula("H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3")
H_jw = hamiltoniano_qubits(m)
H_red = reducir_dos_qubits(hamiltoniano_qubits(m, destruccion=destruccion_paridad), m)

combinaciones = [("Jordan-Wigner + químico", ansatz_ucc(m), H_jw, None)]
for reps in [1, 2, 3]:
    ansatz = efficient_su2(6, reps=reps)
    x0 = np.random.default_rng(1).uniform(-0.1, 0.1, ansatz.num_parameters)
    combinaciones.append((f"paridad + genérico reps={reps}", ansatz, H_red, x0))

for nombre, ansatz, H, x0 in combinaciones:
    resultado, historial = vqe(ansatz, H, x0=x0, maxiter=5000)
    error = 1000 * (resultado.fun - m["e_fci"])
    grupos = len(H.group_commuting(qubit_wise=True))
    print(f"{nombre:28} qubits = {ansatz.num_qubits}  parámetros = {ansatz.num_parameters:2}  "
          f"CNOTs = {contar_cnots(ansatz):3}  grupos = {grupos}  mediciones = {resultado.nfev:4}  error = {error:.3f} mHa")

**Respuesta.** Solo el ansatz químico llega a precisión química, pero su circuito tiene cientos de CNOTs, demasiadas para las computadoras cuánticas actuales. Los ansatz genéricos tienen muy pocas CNOTs y usan 6 qubits, pero quedan lejos de la energía exacta, incluso con varias capas, porque el optimizador se queda atascado. Algunas ideas para probar son quedarse solo con los saltos más importantes del ansatz químico, empezar el ansatz genérico desde el estado de Hartree-Fock o cambiar de optimizador.

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
def energia_en_subespacio(H, bitstrings, devolver_estado=False):
    """Diagonaliza H usando solo las configuraciones de la lista de bitstrings."""
    matriz = H.to_matrix(sparse=True)
    indices = sorted({int(b, 2) for b in bitstrings})
    # Nos quedamos con las filas y columnas de esas configuraciones
    chica = matriz[indices][:, indices].toarray()
    autovalores, autovectores = np.linalg.eigh(chica)
    if not devolver_estado:
        return autovalores[0]
    # Armamos el estado completo, con amplitud cero en las configuraciones que no usamos
    estado = np.zeros(2 ** H.num_qubits, dtype=complex)
    estado[indices] = autovectores[:, 0]
    return autovalores[0], estado


def es_valida(bitstring, m):
    """Indica si el bitstring tiene la cantidad correcta de electrones de cada spin."""
    norb = m["norb"]
    n_alfa, n_beta = m["nelec"]
    return bitstring[norb:].count("1") == n_alfa and bitstring[:norb].count("1") == n_beta

La siguiente celda es código de la clase que usan los ejercicios que siguen.

In [ ]:
m = molecula("H 0 0 0; H 0 0 1.5; H 0 0 3; H 0 0 4.5")
H = hamiltoniano_qubits(m)
print("Error de Hartree-Fock =", 1000 * (m["e_hf"] - m["e_fci"]), "mHa")

# Un circuito aproximado, con solo 40 pasos de VQE
ansatz = ansatz_ucc(m)
resultado, historial = vqe(ansatz, H, maxiter=40)
print("Error del VQE corto =", 1000 * (resultado.fun - m["e_fci"]), "mHa")

# Fijamos los parámetros y agregamos las mediciones
circuito = ansatz.assign_parameters(resultado.x)
circuito.measure_all()

sampler = AerSampler(seed=11)
cuentas = sampler.run([circuito], shots=1000).result()[0].data.meas.get_counts()

validas = [b for b in cuentas if es_valida(b, m)]
energia = energia_en_subespacio(H, validas)
print("Configuraciones distintas =", len(validas))
print("Error de muestrear y diagonalizar =", 1000 * (energia - m["e_fci"]), "mHa")

## Ejercicio 7

a. Usando el mismo circuito, calcula la energía con muestrear y diagonalizar para 10, 30, 100, 300, 1000 y 3000 shots. Para cada caso, guarda la cantidad de configuraciones válidas distintas y el error en mHa.

b. Grafica el error en función de la cantidad de shots, con escala logarítmica en el eje horizontal (```plt.xscale("log")```), junto con una línea horizontal en la precisión química y otra en el error del VQE corto.

c. ¿La energía queda alguna vez por debajo de la energía exacta? ¿Por qué?

In [ ]:
lista_shots = [10, 30, 100, 300, 1000, 3000]
errores = []

for shots in lista_shots:
    cuentas = sampler.run([circuito], shots=shots).result()[0].data.meas.get_counts()
    validas = [b for b in cuentas if es_valida(b, m)]
    error = 1000 * (energia_en_subespacio(H, validas) - m["e_fci"])
    errores.append(error)
    print(f"shots = {shots:4}  configuraciones = {len(validas):2}  error = {error:.3f} mHa")

In [ ]:
plt.plot(lista_shots, errores, "o-", label="muestrear y diagonalizar")
plt.axhline(1000 * (resultado.fun - m["e_fci"]), color="red", linestyle="--", label="VQE corto")
plt.axhline(1.6, color="green", linestyle="--", label="precisión química")
plt.xscale("log")
plt.xlabel("shots")
plt.ylabel("error (mHa)")
plt.legend()
plt.show()

**Respuesta.** Con más shots aparecen más configuraciones y el error baja. Con muy pocos shots, la energía puede ser incluso peor que la del VQE corto, porque todavía faltan configuraciones importantes. La energía nunca queda por debajo de la energía exacta, porque la energía en un subespacio es siempre una cota superior, por el principio variacional.

## Ejercicio 8

a. Escribe una función ```agregar_ruido(bitstrings, p, rng)``` que reciba una lista de bitstrings y devuelva una lista nueva donde cada bit se cambió (de 0 a 1 o de 1 a 0) con probabilidad ```p```. Usen ```rng = np.random.default_rng(0)``` para generar números aleatorios con ```rng.random()```.

b. Toma 1000 muestras del circuito con ```sampler.run([circuito], shots=1000).result()[0].data.meas.get_bitstrings()```, que devuelve la lista de todos los bitstrings medidos. Para $p$ igual a 0, 0,02, 0,05 y 0,1, agrega ruido y muestra la fracción de bitstrings válidos y el error de la energía usando solo los válidos.

c. La probabilidad de que ningún bit de un bitstring cambie es $(1-p)^n$, donde $n$ es la cantidad de qubits. Calcula esa probabilidad para $p = 0{,}02$ y $n$ igual a 8, 50 y 100. ¿Qué les dice eso sobre descartar los bitstrings inválidos en moléculas grandes?

In [ ]:
def agregar_ruido(bitstrings, p, rng):
    nuevos = []
    for b in bitstrings:
        bits = [("1" if c == "0" else "0") if rng.random() < p else c for c in b]
        nuevos.append("".join(bits))
    return nuevos

In [ ]:
rng = np.random.default_rng(0)
muestras = sampler.run([circuito], shots=1000).result()[0].data.meas.get_bitstrings()

for p in [0, 0.02, 0.05, 0.1]:
    ruidosas = agregar_ruido(muestras, p, rng)
    validas = [b for b in ruidosas if es_valida(b, m)]
    error = 1000 * (energia_en_subespacio(H, validas) - m["e_fci"])
    print(f"p = {p:4}  válidos = {len(validas) / len(ruidosas):.0%}  error = {error:.3f} mHa")

In [ ]:
for n in [8, 50, 100]:
    print(f"n = {n:3}  probabilidad sin errores = {(1 - 0.02) ** n:.1%}")

**Respuesta.** Con 8 qubits y poco ruido, la fracción de bitstrings válidos baja, pero quedan suficientes y la energía casi no cambia. Con $p = 0{,}1$, en cambio, menos de la mitad de los bitstrings son válidos y el error empeora. Con 100 qubits, en cambio, solo un 13 % de los bitstrings llegaría sin ningún error, así que descartar los inválidos desperdiciaría casi todas las muestras. Por eso hace falta la recuperación de configuraciones.

## Ejercicio 9

a. Escribe una función ```combinar_alfa_beta(bitstrings, m)``` que separe las partes $\alpha$ y $\beta$ de una lista de configuraciones válidas y devuelva todas las combinaciones posibles. Recuerden que la parte $\alpha$ son los últimos ```m["norb"]``` caracteres del bitstring.

b. Para 30, 100 y 300 shots, compara la cantidad de configuraciones y el error usando solo las configuraciones medidas y usando todas las combinaciones.

c. Para $H_4$ hay solo $\binom{4}{2} = 6$ partes $\alpha$ posibles y 6 partes $\beta$ posibles. ¿Qué pasa cuando aparecen todas? ¿Pasaría lo mismo con una molécula grande?

In [ ]:
def combinar_alfa_beta(bitstrings, m):
    norb = m["norb"]
    alfas = {b[norb:] for b in bitstrings}
    betas = {b[:norb] for b in bitstrings}
    return [beta + alfa for beta in betas for alfa in alfas]

In [ ]:
for shots in [30, 100, 300]:
    cuentas = sampler.run([circuito], shots=shots).result()[0].data.meas.get_counts()
    validas = [b for b in cuentas if es_valida(b, m)]
    combinadas = combinar_alfa_beta(validas, m)
    error_solas = 1000 * (energia_en_subespacio(H, validas) - m["e_fci"])
    error_combinadas = 1000 * (energia_en_subespacio(H, combinadas) - m["e_fci"])
    print(f"shots = {shots:3}  medidas = {len(validas):2} ({error_solas:8.3f} mHa)  "
          f"combinadas = {len(combinadas):2} ({error_combinadas:8.3f} mHa)")

**Respuesta.** Combinando las partes $\alpha$ y $\beta$, el subespacio crece mucho y el error baja muchísimo con la misma cantidad de shots. Cuando aparecen las 6 partes $\alpha$ y las 6 partes $\beta$, las combinaciones son las 36 configuraciones posibles y obtenemos la energía exacta. Con una molécula grande no pasa eso, porque la cantidad de partes posibles es enorme, y el subespacio combinado sigue siendo una fracción muy chica de todas las configuraciones.

## Ejercicio 10

Sigan con la cadena $H_4$ estirada.

a. Calcula la energía de CCSD con ```cc.CCSD(m["hf"]).run().e_tot``` y compárala con la energía de Hartree-Fock y con la exacta. ¿Qué tiene de raro el resultado de CCSD?

b. Sin mirar la energía exacta, calcula la energía de muestrear y diagonalizar (combinando $\alpha$ y $\beta$) para 30, 100, 300 y 1000 shots. ¿El resultado converge?

c. Con ```energia_en_subespacio(H, configuraciones, devolver_estado=True)``` pueden obtener también el estado. Calcula la varianza de la energía para el estado de muestrear y diagonalizar con 30 y con 1000 shots, y también para el estado de Hartree-Fock. Pueden usar ```H.to_matrix(sparse=True)``` para obtener la matriz $M$, y calcular $\left<\hat{H}\right> = \psi^\dagger M \psi$ y $\left<\hat{H}^2\right> = (M\psi)^\dagger (M\psi)$. ¿Qué relación encuentran entre la varianza y el error?

In [ ]:
ccsd = cc.CCSD(m["hf"]).run()

print("Hartree-Fock =", m["e_hf"])
print("CCSD         =", ccsd.e_tot)
print("Exacta       =", m["e_fci"])
print("Error de CCSD =", 1000 * (ccsd.e_tot - m["e_fci"]), "mHa")

In [ ]:
for shots in [30, 100, 300, 1000]:
    cuentas = sampler.run([circuito], shots=shots).result()[0].data.meas.get_counts()
    validas = [b for b in cuentas if es_valida(b, m)]
    print(f"shots = {shots:4}  energía = {energia_en_subespacio(H, combinar_alfa_beta(validas, m)):.6f} Ha")

In [ ]:
matriz = H.to_matrix(sparse=True)


def varianza(estado):
    h_psi = matriz @ estado
    energia = np.vdot(estado, h_psi).real
    return np.vdot(h_psi, h_psi).real - energia ** 2


for shots in [30, 1000]:
    cuentas = sampler.run([circuito], shots=shots).result()[0].data.meas.get_counts()
    validas = [b for b in cuentas if es_valida(b, m)]
    energia, estado = energia_en_subespacio(H, combinar_alfa_beta(validas, m), devolver_estado=True)
    print(f"shots = {shots:4}  varianza = {varianza(estado):.2e}  error = {1000 * (energia - m['e_fci']):.3f} mHa")

estado_hf = Statevector.from_label("00110011").data
print(f"Hartree-Fock  varianza = {varianza(estado_hf):.2e}  error = {1000 * (m['e_hf'] - m['e_fci']):.3f} mHa")

**Respuesta.** CCSD da una energía por debajo de la energía exacta. No es un error del cálculo, es que CCSD no es un método variacional, así que su energía no es una cota superior. Muestrear y diagonalizar, en cambio, converge desde arriba y deja de cambiar a partir de unos 100 shots. La varianza acompaña al error, es grande para Hartree-Fock y prácticamente cero cuando el subespacio ya contiene todas las configuraciones importantes.